In [ ]:
# This is a sample Python script.

# Press Shift+F10 to execute it or replace it with your code.
# Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.


import dash
import dash_html_components as html
import dash_core_components as dcc
from dash.dependencies import Input, Output
import dash_bootstrap_components as dbc

from dash import dash_table
from dash_table.Format import Format
from dash.dash_table import FormatTemplate
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import silhouette_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.cluster import KMeans

import statsmodels.api as sm
from scipy.stats import pointbiserialr
from scipy.stats import kruskal

from tslearn.clustering import TimeSeriesKMeans
#import warnings
#warnings.filterwarnings('ignore')

from tqdm import tqdm

import xgboost as xgb
from xgboost import plot_importance

#=========================================================#
#                    Preparacion previa                   #
#=========================================================#
Tabla = pd.read_csv("C:/Users/guy3h/Downloads/dataset_ventas.csv")

Datos = Tabla.rename(columns={ "ID Producto": "ProductoId",
                               "ID Tienda" : "TiendaId",
                               "Fecha de Venta" : "FechaVenta",
                               "Cantidad Vendida" : "CantidadVendida",
                               "Precio Unitario" : "PrecioUnitario",
                               "Categoría de Producto" : "CategoriaProducto",
                               "Ubicación de Tienda" : "UbicacionTienda",
                               "Promoción o Descuento" : "PromocionDescuento",
                               "Estacionalidad" : "Estacionalidad",
                               "Método de Pago" : "MetodoPago",
                               "Nivel de Inventario" : "NivelInventario"
                               })

# Reordenar las Columnas
Datos = Datos[[ 'TiendaId', 'ProductoId', 'FechaVenta', 'CantidadVendida',
                'PrecioUnitario', 'CategoriaProducto', 'UbicacionTienda',
                'PromocionDescuento', 'Estacionalidad', 'MetodoPago',
                'NivelInventario'  ]]

# Cambiamos los datos en la columna Fecha-Venta al tipo "Fecha"
Datos['FechaVenta'] = pd.to_datetime(Datos['FechaVenta'], format='%Y-%m-%d')
Datos['FechaVenta'] = Datos['FechaVenta'].dt.normalize()   # remove Hours,Minutes,Seconds from DateTime

Quant  = ['CantidadVendida','PrecioUnitario','NivelInventario']  # Variables Cuantitativas
Cualit = ['PromocionDescuento', 'Estacionalidad', 'MetodoPago']  # Variables Cualitativas

Tienda_orden = Datos['TiendaId'].copy().unique().tolist()
Tienda_orden.sort(key=lambda fname: int(fname.split('_')[1]))
Producto_orden = Datos['ProductoId'].copy().unique().tolist()
Producto_orden.sort(key=lambda fname: int(fname.split('_')[1]))

COLORES = ['steelblue', 'darkorange', 'green', 'firebrick', 'yellow', 'darkviolet', 'aqua']
COLORES2 = ['green','red','darkorange']

#-----------------------------------------#
#          Tabla Intermedia               #
#-----------------------------------------#

METODOPAGO = {'Efectivo':1, 'Crédito':2, 'Tarjeta':3}
def MOBO_Indicadores():
    MOBO = pd.DataFrame({},
                        index=[i + 1 for i in range(500)],
                        columns=['Tienda','Producto','PrecioUnitario','PromocionDescuento',
                                 'Estacionalidad','MetodoPago','NivelInventario'] )
    i = 1
    for t in Datos['TiendaId'].unique():
        for p in Datos['ProductoId'].unique():
            TP = Datos[(Datos['TiendaId'] == t) & (Datos['ProductoId'] == p)].sort_values(by='FechaVenta')

            #CantPrecio = TP[['CantidadVendida', 'PrecioUnitario', 'PromocionDescuento']]
            #r1 = silhouette_score(CantPrecio[['CantidadVendida', 'PrecioUnitario']].values, CantPrecio['PromocionDescuento'])

            # r1 = np.corrcoef(TP['CantidadVendida'], TP['PrecioUnitario'])[0,1]
            # r2 = np.corrcoef(TP['CantidadVendida'], TP['PromocionDescuento'])[0,1]
            # r3 = np.corrcoef(TP['CantidadVendida'], TP['Estacionalidad'])[0,1]
            #
            # #CantMetod = TP[['CantidadVendida', 'PrecioUnitario', 'MetodoPago']]
            # #r4 = silhouette_score(CantMetod[['CantidadVendida', 'PrecioUnitario']].values, CantMetod['MetodoPago'])
            #
            # r4 = np.corrcoef(TP['CantidadVendida'], TP['MetodoPago'].map(METODOPAGO))[0,1]
            # r5 = np.corrcoef(TP['CantidadVendida'], TP['NivelInventario'])[0,1]

            r1 = np.corrcoef(TP['CantidadVendida'], TP['PrecioUnitario'])[0,1]
            r2 = pointbiserialr(TP['CantidadVendida'], TP['PromocionDescuento'])[1]
            r3 = pointbiserialr(TP['CantidadVendida'], TP['Estacionalidad'])[1]

            G1 = TP[TP['MetodoPago'] == 'Efectivo']['CantidadVendida']
            G2 = TP[TP['MetodoPago'] == 'Crédito']['CantidadVendida']
            G3 = TP[TP['MetodoPago'] == 'Tarjeta']['CantidadVendida']

            r4 = kruskal(G1, G2, G3)[1]
            r5 = np.corrcoef(TP['CantidadVendida'], TP['NivelInventario'])[0,1]

            #---------- Tienda y Producto -------#
            tO = t.split('_')[1]
            pO = p.split('_')[1]

            MOBO.loc[i] = [int(tO),int(pO), np.round(r1,4),
                           np.round(r2,4),np.round(r3,4),np.round(r4,4),np.round(r5,4)]
            i += 1

    MOBO = MOBO.sort_values(by=['Tienda','Producto'], ascending=True)
    MOBO.index = [i + 1 for i in range(500)]
    return(MOBO)

MOBO = MOBO_Indicadores()

#-----------------------------------------#
#           Funcion Intermedia            #
#-----------------------------------------#
def mi_Prediccion(X,xgb_reg, n):
    # prediccion para los proximos (n) dias
    Hist = X.copy()

    for tO in pd.date_range(start='2024-01-01', freq='D', periods=n):
        r1 = Hist[-7:].count(0)
        r2 = Hist[-30:].count(0)
        r3 = len(Hist) - np.where(np.array(Hist) != 0)[0][-1]
        r4 = np.sum(Hist[-7:])
        r5 = np.sum(Hist[-30:])
        caracteristicas = [tO.day, tO.month, tO.year, tO.weekday(), r1, r3, r4]

        pred = xgb_reg.predict(np.array([caracteristicas]))
        pred = np.abs(np.floor(pred))
        pred = pred[0]
        Hist.append(pred)

    Pred = Hist[-730:]    # los ultimos 2 años
    return (Pred)


#=========================================================#
#                         DASHBOARD                       #
#=========================================================#
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = html.Div(children=[
    # -----------------------------------------------------------------------------#
    html.Br(),
    html.Br(),


    dbc.Row([
        html.H3("MOBO : Analisis Exploratorio",
                style={'textAlign': 'center',
                       'color': 'white',
                       'border-radius': '20px',
                       'backgroundColor': 'steelblue'}),  # steelblue
    ]),

    html.Br(),
    html.Br(),
    html.Hr(),

    #--------------------------------------------------------------------------#
    #                FILA 1 = Histograma, Tabla, Grafica Dispersion            #
    #--------------------------------------------------------------------------#


    dbc.Row([
        #-------------------------- Histograma -----------------------------#
        dbc.Col([
            dbc.Row([
                html.Label(['Variable'], style={'marginLeft': '160px'}),
                dcc.Dropdown(
                    id='Variable_X',
                    placeholder='Seleccionar la Variable a analizar',
                    options=MOBO.columns[2:],
                    value='PrecioUnitario',
                    # multi=True,
                    clearable=False,
                    style={'font-size': '14px',
                           'textAlign': 'center',
                           'border-radius': '20px',
                           'text': 'black'}),
            ], style={'marginLeft': '10px',
                      'marginRight': '10px',
                      'marginBottom': '10px'}),
            dbc.Row([
                dcc.Graph(id='Histograma_1',
                          style={#'width': '500px',
                                 'height': '300px',
                                 'marginTop':'0px',})
            ], style={ 'marginTop': '0px',
                       'marginLeft': '2px', 'marginRight': '2px',
                       'border-radius': '20px', 'backgroundColor': 'white'}),
        ], style={ 'marginLeft': '2px', 'marginRight': '2px'},
           #width=4
        ),

        #--------------------------- Tablita -------------------------------#
        dbc.Col([
            dbc.Row([
                html.Label(['Analizar Bin'], style={'marginLeft': '140px'}),
                dcc.Dropdown(
                    id='Bin_1',
                    placeholder='Seleccionar Bin',
                    value=1,
                    #multi=True,
                    clearable=False,
                    style={'font-size': '14px',
                           'textAlign': 'center',
                           'border-radius': '20px',
                           'text': 'black'}),  # '#EAEDED'
            ], style={'marginLeft': '10px',
                      'marginRight': '10px',
                      'marginBottom': '10px'}),
            dbc.Row([
                dash_table.DataTable(
                    id='Tablita',
                    fixed_rows={'headers': True},
                    #page_size=20,
                    style_table={'height': '280px', 'overflowY': 'auto', "marginTop": "10px", "marginBottom": "10px"},
                    style_header={'textAlign': 'center', 'backgroundColor': '#305D91',
                                  'color': '#FFFFFF', 'height': 'auto',
                                  'overflow': 'auto'
                                  },  # 'overflow': 'inherit', 'textOverflow': 'ellipsis',
                    # 'whiteSpace': 'normal', "fontSize": "10px",
                    style_cell={'textAlign': 'center',
                                'font_size': '13px', 'minWidth': '60px',
                                'height': 'auto', 'whiteSpace': 'normal'},
                    # 'whiteSpace': 'normal', , 'overflow': 'hidden'
                )
            ], style={'border-radius': '20px',
                      'backgroundColor': 'white',
                      #'border' : '0.5px solid',
                      'marginLeft': '10px', 'marginRight': '10px'}),
        ], style={ 'marginLeft': '2px', 'marginRight': '2px'},
           #width=3
        ),

        #----------------------- Grafica Dispersion ------------------------#
        dbc.Col([
            dbc.Row([
                html.Label(['Combinacion Tienda-Producto'], style={'marginLeft': '70px'}),
                dcc.Dropdown(
                    id='Tiend_Prod',
                    placeholder='Seleccionar la combinacion Tienda-Producto a visualizar',
                    clearable=False,
                    style={'font-size': '14px',
                           'textAlign': 'center',
                           'border-radius': '20px',
                           'text': 'black'}),
            ], style={'marginLeft': '10px',
                      'marginRight': '10px',
                      'marginBottom': '10px'}),

            dbc.Row([
                dcc.Graph(id='Rel_Linear',
                          style={ #'width': '500px',
                                 'height': '300px',
                                 'marginTop': '0px', })
            ], style={'marginTop': '0px',
                      'marginLeft': '5px', 'marginRight': '15px',
                      'border-radius': '20px', 'backgroundColor': 'white'},
            ),
        ], style={ 'marginLeft': '2px', 'marginRight': '2px'},
           #width=4
        ),

    ], style={'marginLeft': '20px', 'marginRight': '20px'}),

    html.Br(),
    html.Br(),
    html.Hr(),
    html.Br(),

    #------------------------------------------------------------------------#
    #                          FILA 2 = CLUSTERING                           #
    #------------------------------------------------------------------------#
    dbc.Row([
        dbc.Col([
            dbc.Row([
                #-------------------- (1) Numero Cluster -----------------------------#
                dbc.Col([
                    html.Label(['Numero de Cluster'], style={'marginLeft': '40px'}),
                    dcc.Dropdown(
                        id='n_Cluster',
                        placeholder='Seleccionar el numero de Cluster',
                        options=[i for i in range(2,Datos['TiendaId'].nunique()-1)],
                        value=2,
                        clearable=False,
                        style={'font-size': '12px',
                               'textAlign': 'center',
                               'border-radius': '20px',
                               'text': 'black'}),
                ], width=4),
                # ------------------- (2) Etiqueta de Cluster ------------------------#
                dbc.Col([
                    html.Label(['Etiqueta de Cluster'], style={'marginLeft': '40px'}),
                    dcc.Dropdown(
                        id='Cluster_Label',
                        placeholder='Seleccionar el Cluster a analizar:',
                        value=1,
                        clearable=False,
                        style={'font-size': '12px',
                               'textAlign': 'center',
                               'border-radius': '20px',
                               'text': 'black'}),
                ], width=4),
                #-------------------- (3) Elementos del Cluster ----------------------#
                dbc.Col([
                    html.Label(['Elementos del Cluster'], style={'marginLeft': '30px'}),
                    dcc.Dropdown(
                        id='Cluster_Elements',
                        placeholder='Seleccionar los elementos del Cluster a visualizar',
                        multi=True,
                        clearable=False,
                        style={'font-size': '12px',
                               'textAlign': 'center',
                               'border-radius': '20px',
                               'text': 'black'}),
                ], width=4),
            ], style={'marginLeft': '20px',
                      'marginRight': '20px',
                      'marginBottom': '10px'}),
            #-------------------- Grafica de Clusters ------------------------#
            dbc.Row([
                dcc.Graph(id='Cluster_Tiendas',
                          style={'height': '300px' })
            ], style={'marginLeft': '5px', 'marginRight': '15px',
                      'border-radius': '20px', 'backgroundColor': 'white'},
            ),
        ], width=8),
        #-------------------- Tabla con Cluster ------------------------#
        dbc.Col([
            html.Label("CLUSTER de TIENDAS",
                       style={'textAlign': 'center',
                              'color': 'white',
                              'font-weight': 'bold',
                              'border-radius': '5px',
                              'backgroundColor': 'darkorange',
                              'marginLeft': '120px'}),
            dash_table.DataTable(
                id='Tabla_Clusters',
                fixed_rows={'headers': True},
                page_size=36,
                style_table={'height': '300px', "marginTop": "10px", "marginBottom": "10px"},
                style_header={'textAlign': 'center', 'backgroundColor': '#305D91',
                              'color': '#FFFFFF', 'height': 'auto',
                              'overflow': 'auto'
                              },  # 'overflow': 'inherit', 'textOverflow': 'ellipsis',
                # 'whiteSpace': 'normal', "fontSize": "10px",
                style_cell={'textAlign': 'center',
                            'font_size': '13px', 'minWidth': '90px',
                            'height': 'auto', 'whiteSpace': 'normal'},
                # 'whiteSpace': 'normal', , 'overflow': 'hidden'
            )
        ], style={'marginLeft': '5px', 'marginRight': '15px',
                  'border-radius': '20px', 'backgroundColor': 'white'},),
    ], style={'marginLeft': '20px', 'marginRight': '20px'}),

    html.Br(),
    html.Hr(),
    html.Br(),

    #------------------------------------------------------------------------#
    #                               Drop-Down Menu                           #
    #------------------------------------------------------------------------#
    dbc.Row([
        #--------------------------- Tienda ------------------------#
        dbc.Col([
            html.Label(['Tienda'], style={'marginLeft': '20px'}),
            dcc.Dropdown(
                id='Tienda',
                placeholder='Seleccionar Tienda',
                options=Tienda_orden,
                value='Tienda_1',
                #multi=True,
                clearable=False,
                style={'font-size': '12px',
                       'border-radius': '20px',
                       'text': 'black',
                       'backgroundColor': 'white'}),  # '#EAEDED'
        ], width=3),

        #--------------------------- Producto ----------------------#
        dbc.Col([
            html.Label(['Producto'], style={'marginLeft': '20px'}),
            dcc.Dropdown(
                id='Producto',
                placeholder='Seleccionar Producto',
                options=Producto_orden,
                value='Producto_1',
                clearable=False,
                style={'font-size': '12px',
                       'border-radius': '20px',
                       'text': 'black',
                       'backgroundColor': 'white'}),  # '#EAEDED'
        ], width=3),
    ], style={'marginLeft': '10px', 'marginRight': '10px'}),

    html.Br(),
    html.Br(),

    #-------------------------------------------------------------------------#
    #                                 TABLA                                   #
    #-------------------------------------------------------------------------#
    dbc.Row([
        dbc.Col([
            dash_table.DataTable(
                id='Tabla',
                fixed_rows={'headers': True},
                page_size=36,
                style_table={'height': '300px', "marginTop":"10px", "marginBottom":"10px"},
                style_header={'textAlign': 'center', 'backgroundColor': '#305D91',
                              'color': '#FFFFFF', 'height': 'auto',
                              'overflow': 'auto'
                              },  # 'overflow': 'inherit', 'textOverflow': 'ellipsis',
                                  # 'whiteSpace': 'normal', "fontSize": "10px",
                style_cell={'textAlign': 'right',
                            'font_size': '13px', 'minWidth': '90px',
                            'height': 'auto', 'whiteSpace': 'normal'},
                # 'whiteSpace': 'normal', , 'overflow': 'hidden'
            )
        ], style={'border-radius': '20px',
                  #'backgroundColor': 'lightgrey',
                  'border' : '0.5px solid',
                  'marginLeft': '10px', 'marginRight': '10px'})
    ], style={'marginLeft': '15px', 'marginRight': '15px'}),


    html.Br(),
    html.Br(),

    #--------------------------------------------------------------------------#
    #                             FILA debajo de Tabla                         #
    #--------------------------------------------------------------------------#

    dbc.Row([
        #-------------------------- Discriminante -----------------------------#
        dbc.Col([
            dbc.Row([
                dbc.Col([
                    html.Label(['Variable 1'], style={'marginLeft': '10px'}),
                    dcc.Dropdown(
                        id='X_1',
                        placeholder='Seleccionar la Variable para el eje X',
                        value='CantidadVendida',
                        options=Quant,
                        clearable=False,
                        style={'font-size': '12px',
                               'textAlign': 'center',
                               'border-radius': '20px',
                               'text': 'black'}),
                ]),
                dbc.Col([
                    html.Label(['Variable 2'], style={'marginLeft': '10px'}),
                    dcc.Dropdown(
                        id='X_2',
                        placeholder='Seleccionar la Variable para el eje Y',
                        value='PrecioUnitario',
                        clearable=False,
                        style={'font-size': '12px',
                               'textAlign': 'center',
                               'border-radius': '20px',
                               'text': 'black'}),
                ]),
                dbc.Col([
                    html.Label(['Discriminante'], style={'marginLeft': '10px'}),
                    dcc.Dropdown(
                        id='X_3',
                        placeholder='Seleccionar la Variable con la que se etiqueta',
                        value='PromocionDescuento',
                        options=Cualit,
                        clearable=False,
                        style={'font-size': '12px',
                               'textAlign': 'center',
                               'border-radius': '20px',
                               'text': 'black'}),
                ]),
            ], style={'marginLeft': '10px',
                      'marginRight': '10px'}),

            #------------------ Grafica Discriminante ------------#
            dbc.Row([
                dcc.Graph(id='Discriminante',
                          style={'height': '355px',
                                 'marginTop': '0px', })
            ], style={'marginTop': '35px',
                      'marginLeft': '2px', 'marginRight': '2px',
                      'border-radius': '20px', 'backgroundColor': 'white'}),
        ], style={'marginLeft': '2px',
                  'marginRight': '2px',
                  'marginBottom': '10px'},
           width=4
        ),
        #------------------- GRAFICA Serie --------------------#
        dbc.Col([
            #------------------- Parametros Arbol --------------------#
            dbc.Row([
                #------------------- Parametro 01 --------------------#
                dbc.Col([
                    html.Label(['Profundidad Arbol'], style={'marginLeft': '20px'}),
                    # dcc.Slider(id='Slider_1',
                    #            min=1,
                    #            max=20,
                    #            step=1,
                    #            marks={i:i for i in range(21)}),
                    dcc.Dropdown(
                        id='Param_1',
                        placeholder='Seleccionar la Profundidad del arbol',
                        value=10,
                        options=[i for i in range(1,21)],
                        clearable=False,
                        style={'font-size': '12px',
                               'textAlign': 'center',
                               'border-radius': '20px',
                               'text': 'black'}),
                ], style={'border-radius': '40px'},
                   width=3),
                #------------------- Parametro 02 --------------------#
                dbc.Col([
                    html.Label(['Numero Estimad'], style={'marginLeft': '20px'}),
                    # dcc.Slider(id='Slider_2',
                    #            min=1,
                    #            max=20,
                    #            step=1,
                    #            marks={i:i for i in range(21)}),
                    dcc.Dropdown(
                        id='Param_2',
                        placeholder='Seleccionar el Numero de Estimadores del arbol',
                        value=5,
                        options=[i for i in range(1, 21)],
                        clearable=False,
                        style={'font-size': '12px',
                               'textAlign': 'center',
                               'border-radius': '20px',
                               'text': 'black'}),
                ], style={'border-radius': '40px'},
                   width=3),
            ], style={'marginLeft': '80px',
                      'marginRight': '20px'}),

            #----------------- Serie de Tiempo -------------#
            dbc.Row([
                dcc.Graph(id='SerieTiempo',
                          style={  # 'width': '500px',
                              'height': '350px',
                              'marginTop': '5px', })
            ], style={'marginTop': '35px',
                      'border-radius': '20px',
                      'backgroundColor': 'white'}),
        ], style={'marginLeft': '5px', 'marginRight': '5px',
                  'border-radius': '20px'},
            #width=6,
        ),
    ], style={'marginLeft': '20px', 'marginRight': '20px'}),

    html.Br(),
    #--------------------------------------------------------------------------#
    #                             Importancia Variables                        #
    #--------------------------------------------------------------------------#
    dbc.Row([
        dbc.Col([

        ], width=4),

        dbc.Col([
            dbc.Row([
                dcc.Graph(id='Importancia_Var',
                          style={'height': '250px',
                                 'marginTop': '10px'})
            ], style={'border-radius': '20px',
                      'backgroundColor': 'white'}),
        ], style={ 'marginRight': '230px'},
        ),
    ]),

    html.Br(),
    html.Br(),
    html.Hr(),
    html.Br(),
    html.Br(),
    html.Br(),
], style={'backgroundColor': '#EAEDED'})


#******************************************************************************************************************************************************#
#******************************************************************************************************************************************************#


# =======================================#
#                 Tienda                 #
# =======================================#
# @app.callback(Input(component_id='Tienda', component_property='value'))
# def Listar_Tiendas(r):
#     return None



#========================================#
#               Histograma               #
#========================================#
@app.callback(Output(component_id='Histograma_1', component_property='figure'),
              Output(component_id='Bin_1', component_property='options'),
              Input(component_id='Variable_X', component_property='value'),
              Input(component_id='Bin_1', component_property='value'))
def Crear_Histograma(variable,menu):
    #bins1 = {'start':-1, 'end':1, 'size':0.05}  #[i for i in np.arange(0, 1.1, 0.1)]
    #bins2 = {'start': -1, 'end': 1, 'size': 0.1}


    #---------------------- Indicador de Bins ------------------------#
    # if variable in ['Cant-Precio','Cant-Metod']:
    #     bins1 = [np.round(i,2) for i in np.arange(np.floor(MOBO[variable].min()/0.05)*0.05,
    #                                                np.floor(MOBO[variable].max()/0.05)*0.05+0.05*2, 0.05)]
    #                                                # +0.05*2 es para :
    #                                                # 1.- cancelar el efecto np.floor
    #                                                # 2.- agregar el margen porque np.arange llega al (n-1)
    #     MOBO['Bin'] = MOBO[variable].apply(lambda x: np.floor(x/0.05))
    #     counts1, bins11 = np.histogram(MOBO[variable].values, bins=bins1)
    # else :
    #     bins2 = [np.round(i,2) for i in np.arange( np.floor(MOBO[variable].min()/0.1)*0.1,
    #                                                np.floor(MOBO[variable].max()/0.1)*0.1+0.1*2, 0.1)]
    #     MOBO['Bin'] = MOBO[variable].apply(lambda x: np.floor(x/0.1))
    #     counts2, bins22 = np.histogram(MOBO[variable].values, bins=bins2)

    bins1 = [np.round(i,2) for i in np.arange( np.floor(MOBO[variable].min()/0.1)*0.1,
                                              np.floor(MOBO[variable].max()/0.1)*0.1+0.1*2, 0.1)]
    MOBO['Bin'] = MOBO[variable].apply(lambda x: np.floor(x/0.1))
    counts1, bins2 = np.histogram(MOBO[variable].values, bins=bins1)

    #------------------------------------------------------------------------------#

    min = np.abs( MOBO['Bin'].min() )
    MOBO['Bin'] = MOBO['Bin']+min+1        # Recorremos para que el primer valor sea el (1),
                                           # es decir Grupo=1
    Menu_1 = MOBO.sort_values(by=['Bin'], ascending=True)['Bin'].unique().tolist()

    columns = [ {'name':i, 'id':i} for i in ['Tienda','Producto',variable,'Bin'] ]
    data = MOBO[['Tienda','Producto',variable,'Bin']].to_dict('records')

    #---------------------------- Figure 01 -----------------------------#
    # if variable in ['Cant-Precio','Cant-Metod']:
    #     fig_1 = go.Figure()
    #     fig_1.add_trace(
    #         go.Bar(x=bins1,
    #                y=counts1,
    #                #bingroup=1,
    #                #marker=dict( color='steelblue', line=dict(color='white', width=1 ) )
    #                marker_color=['steelblue' if (i+1)!=menu else 'red' for i in range(len(MOBO['Bin'])) ] )
    #     )
    # else:
    #     fig_1 = go.Figure()
    #     fig_1.add_trace(
    #         go.Bar(x=bins2,
    #                y=counts2,
    #                marker_color=['steelblue' if (i+1)!=menu else 'red' for i in range(len(MOBO['Bin'])) ])
    #     )

    fig_1 = go.Figure()
    fig_1.add_trace(
        go.Bar(x=bins1,
               y=counts1,
               # bingroup=1,
               # marker=dict( color='steelblue', line=dict(color='white', width=1 ) )
               marker_color=['steelblue' if (i + 1) != menu else 'red' for i in range(len(MOBO['Bin']))])
    )

    fig_1.update_layout(
        title='Distribucion de la Correlación entre <br> la Cantidad Vendida y '+variable,
        title_x=0.50,  # center Title
        title_font_size=15,
        xaxis_title='Valor de la Correlacion',
        yaxis_title='Conteo',
        margin={'l': 10, 'r': 30, 't': 50, 'b': 50}
    )

    return([fig_1, Menu_1])


#========================================#
#              Tablita 1                 #
#========================================#
@app.callback(Output(component_id='Tablita', component_property='columns'),
              Output(component_id='Tablita', component_property='data'),
              Output(component_id='Tiend_Prod', component_property='options'),
              Input(component_id='Variable_X', component_property='value'),
              Input(component_id='Bin_1', component_property='value'))
def Crear_Tablita(variable, menu):

    # if variable in ['Cant-Precio','Cant-Metod']:
    #     MOBO['Bin'] = MOBO[variable].apply(lambda x: np.floor(x/0.05))
    # else :
    #     MOBO['Bin'] = MOBO[variable].apply(lambda x: np.floor(x/0.1))
    MOBO['Bin'] = MOBO[variable].apply(lambda x: np.floor(x / 0.1))

    min = np.abs( MOBO['Bin'].min() )
    MOBO['Bin'] = MOBO['Bin']+min+1        # Recorremos para que el primer valor sea el (1),
                                           # es decir Grupo=1

    A = MOBO[ MOBO['Bin']==menu ]
    columns = [ {'name':i, 'id':i} for i in ['Tienda','Producto',variable,'Bin'] ]
    data = A[['Tienda','Producto',variable,'Bin']].to_dict('records')

    Tienda_Producto = 'Tienda_'+A['Tienda'].astype(str)+' '+'Producto_'+A['Producto'].astype(str)
    return ([columns, data, Tienda_Producto])



#========================================#
#            Relacion Linear             #
#========================================#
@app.callback(Output(component_id='Rel_Linear', component_property='figure'),
              Input(component_id='Variable_X', component_property='value'),
              Input(component_id='Tiend_Prod', component_property='value'))
def Relacion_Linear(variable, TP):
    t = TP.split(' ')[0]
    p = TP.split(' ')[1]

    Data = Datos[(Datos['TiendaId']==t) & (Datos['ProductoId']==p)].sort_values(by='FechaVenta').copy()
    Data['MetodoPago'] = Data['MetodoPago'].map({'Efectivo':0,'Tarjeta':1,'Crédito':2})
    opciones = Data[variable].unique().tolist()

    fig = go.Figure()
    if variable in ['PrecioUnitario', 'NivelInventario']:
        fig.add_trace(
            go.Scatter(x=Data['CantidadVendida'],
                       y=Data[variable],
                       #name='',
                       marker={'color': 'steelblue',
                               'size': 10},
                       mode='markers',
                       )
        )
    #------------------------ Regresion Linear --------------------------#

        xi = Data['CantidadVendida'].to_numpy().reshape((-1,1))
        yi = Data[variable]
        model = LinearRegression().fit(xi,yi)
        a,b = model.coef_[0], model.intercept_

        Xi = [Data['CantidadVendida'].min(), Data['CantidadVendida'].max()]
        Yi = [a*k+b for k in Xi]

        fig.add_trace(
            go.Scatter( x=Xi,
                        y=Yi,
                        mode='lines',
                        #name='Points',
                        line={'color':'firebrick'}
        ))
    else:
        for i, opt in enumerate(opciones):
            fig.add_trace(
                go.Scatter(x=Data[Data[variable] == opt]['CantidadVendida'],
                           y=Data[Data[variable] == opt][variable],
                           marker={'color': COLORES[i],
                                   'size': 10},
                           mode='markers',
                           )
            )


    fig.update_layout(
        title='Grafica de Dispersion entre <br> la Cantidad Vendida y '+variable,
        title_font_size=15,
        title_x=0.55,  # center Title
        showlegend=False,
        xaxis_title='Cantidad Vendida',
        yaxis_title=variable,
        margin={'l': 10, 'r': 20, 't': 50, 'b': 50},
    )


    return (fig)


#========================================#
#   Lista para Siguiente Variable X_2    #
#========================================#
@app.callback(Output(component_id='X_2', component_property='options'),
              Input(component_id='X_1', component_property='value'))
def Lista_2(x1):
    Lista = Quant.copy()
    Lista.remove(x1)
    return(Lista)


#========================================#
#              Discriminante             #
#========================================#
@app.callback(Output(component_id='Discriminante', component_property='figure'),
              Input(component_id='Tienda', component_property='value'),
              Input(component_id='Producto', component_property='value'),
              Input(component_id='X_1', component_property='value'),
              Input(component_id='X_2', component_property='value'),
              Input(component_id='X_3', component_property='value'))
def Agrupar_Discriminante(t,p,x1,x2,x3):
    Data = Datos[(Datos['TiendaId'] == t) &
                 (Datos['ProductoId'] == p)].sort_values(by='FechaVenta')

    Data = Data[[x1,x2,x3]].sort_values(by=[x3])
    opciones = Data[x3].unique().tolist()

    fig = go.Figure()
    for i,opt in enumerate(opciones):
        fig.add_trace(
            go.Scatter(x=Data[Data[x3]==opt][x1],
                       y=Data[Data[x3]==opt][x2],
                       name=str(x3)+'='+str(opt),
                       marker={'color': COLORES2[i],
                               'size': 10},
                       mode='markers',
                       )
        )

    #---------------------------------------COLORING-------------------------------------------#
    fig.add_trace(
        go.Scatter(x=[(Data[x1].min()+Data[x1].max())/2,
                       1.05*Data[x1].max()],
                   y=[(Data[x2].min()+Data[x2].max())/2,
                      (Data[x2].min()+Data[x2].max())/2],
                   fill='tozeroy',
                   fillcolor='rgba(0,100,0,0.2)',
                   line={'color': 'rgba(0,0,0,0)',
                         #'dash': 'dash'
                         }
                   )
    )

    #--------------------- fill to-next-Y ------------------#
    fig.add_trace(
        go.Scatter(x=[0.05*Data[x1].min(),
                      (Data[x1].min() + Data[x1].max()) / 2],
                   y=[(Data[x2].min() + Data[x2].max()) / 2,
                      (Data[x2].min() + Data[x2].max()) / 2],
                   line={'color': 'rgba(0,0,0,0)'}
                   )
    )
    fig.add_trace(  ### plotted side-by-side
        go.Scatter(x=[0.05*Data[x1].min(),
                      (Data[x1].min()+Data[x1].max())/2],
                   y=[1.05 * Data[x2].max(),
                      1.05 * Data[x2].max()],
                   fill='tonexty',
                   fillcolor='rgba(102,0,0,0.2)',
                   line={'color': 'rgba(0,0,0,0)'}
                   )
    )
    #-------------------------------------------------------------------------------------#

    fig.update_layout(
        title='',#'Efecto de Promociones-Descuento',
        #title_x=0.1,  # center Title
        xaxis_title='Cantidad Vendida',
        yaxis_title='Precio Unitario',
        showlegend=True,
        legend=dict( orientation="h",
                     yanchor="bottom",
                     y=1.02,
                     xanchor="right",  # You could anchor to the right
                     x=1  # and position it at the right edge
        ),
        margin={'l': 1, 'r': 5, 't': 25, 'b': 30},
        # template='plotly_white',
    )

    return(fig)


#========================================#
#            Cluster Label               #
#========================================#
@app.callback(Output(component_id='Cluster_Label', component_property='options'),
              Output(component_id='Tabla_Clusters', component_property='columns'),
              Output(component_id='Tabla_Clusters', component_property='data'),
              Input(component_id='n_Cluster', component_property='value'))
def Cluster_Labeling(n):
    lista = [i+1 for i in range(n)]
    Data = Datos.copy()
    Data['año'] = Data['FechaVenta'].dt.year
    Data['mes'] = Data['FechaVenta'].dt.month

    Tiendas = Data.pivot_table(index='TiendaId',
                                columns=['año','mes'],
                                values='CantidadVendida',
                                aggfunc='sum')
    Tiendas = pd.DataFrame(Tiendas)
    Tiendas.columns = [i for i in range(1,25)]

    kmeans = TimeSeriesKMeans(n_clusters=n, metric="softdtw", max_iter=10)
    kmeans.fit(Tiendas.values)

    Tiendas_aug = Tiendas.copy()
    Tiendas_aug['Cluster'] = np.array(kmeans.labels_) + 1

    #-------------------------------------------------#
    Grupos = Tiendas_aug.reset_index()
    Grupos = Grupos[['TiendaId','Cluster']]
    columns = [ {'name':i, 'id':i} for i in Grupos.columns ]
    data = Grupos.to_dict('records')

    return(lista, columns, data)


#========================================#
#            Cluster Elements            #
#========================================#
@app.callback(Output(component_id='Cluster_Elements', component_property='options'),
              Input(component_id='Cluster_Label', component_property='value'),
              Input(component_id='Tabla_Clusters', component_property='columns'),
              Input(component_id='Tabla_Clusters', component_property='data'))
def Cluster_Element(cluster,col,dat):
    Data = pd.DataFrame(dat)
    Elementos = Data[Data['Cluster']==cluster]
    Elementos = Elementos['TiendaId']
    return(Elementos)


#========================================#
#        Graficar Cluster Tiendas        #
#========================================#
@app.callback(Output(component_id='Cluster_Tiendas', component_property='figure'),
              Input(component_id='Cluster_Elements', component_property='value'))
def Shop_Cluster(elem):
    Data = Datos.copy()
    Data['año'] = Data['FechaVenta'].dt.year
    Data['mes'] = Data['FechaVenta'].dt.month

    Tiendas = Data.pivot_table(index='TiendaId',
                               columns=['año','mes'],
                               values='CantidadVendida',
                               aggfunc='sum')
    Tiendas = pd.DataFrame(Tiendas)
    Tiendas.columns = [i for i in range(1,25)]
    #print(Tiendas.head())

    fig = go.Figure()
    for n,k in enumerate(elem):
        fig.add_trace(
            go.Scatter(x=Tiendas.columns,
                       y=Tiendas.loc[k,:],
                       name=k,
                       marker={'color': COLORES[n],
                               'size': 10},
                       mode='lines+markers',
                       )
        )

    fig.update_layout(
        title='Grafica Tiendas del mismo Cluster',
        title_x=0.5,  # center Title
        title_font_size=15,
        xaxis_title='Numero de Meses',
        yaxis_title='Cantidad Vendida',
        showlegend=False,
        margin={'l': 10, 'r': 30, 't': 50, 'b': 30},
    )

    return(fig)


#========================================#
#                  TABLA                 #
#========================================#
@app.callback(Output(component_id='Tabla', component_property='columns'),
              Output(component_id='Tabla', component_property='data'),
              Input(component_id='Tienda', component_property='value'),
              Input(component_id='Producto', component_property='value'))
def Crear_Tabla(t,p):
    Data = Datos[ (Datos['TiendaId']==t) & (Datos['ProductoId']==p)].sort_values(by='FechaVenta')
    columns = [ {'name':i, 'id':i} for i in Data.columns ]
    #columns = [{'name': i, 'id': i} for i in Tabla.columns]
    data = Data.to_dict('records')
    return (columns, data)


#========================================#
#            Serie de Tiempo             #
#========================================#
@app.callback(Output(component_id='SerieTiempo', component_property='figure'),
              Output(component_id='Importancia_Var', component_property='figure'),
              Input(component_id='Tienda', component_property='value'),
              Input(component_id='Producto', component_property='value'),
              Input(component_id='Param_1', component_property='value'),
              Input(component_id='Param_2', component_property='value'))
def Serie_Temporal(t,p,slid1,slid2):
    Data = Datos[(Datos['TiendaId'] == t) & (Datos['ProductoId'] == p)].sort_values(by='FechaVenta')

    xO = Data[['FechaVenta', 'CantidadVendida']]
    xO.set_index('FechaVenta', drop=True, inplace=True)

    yO = xO.resample('D').sum().reindex(pd.date_range('2022-01-01', '2023-12-31', freq='D')).fillna(0)

    # -------------------- Serie Temporal ----------------------#

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(x=yO.index,
                   y=yO.values.flatten(),
                   name='Historico Ventas',
                   line={'color': 'steelblue',
                         'width': 1,
                         'dash': 'dashdot'},
                   marker={'color': 'steelblue',
                           'size': 6},
                   mode='lines+markers',
                   )
    )

    fig.update_layout(
        title='Evolucion en el Tiempo',
        title_x=0.2,  # center Title
        xaxis_title='Fecha Venta',
        yaxis_title='Cantidad Vendida',
        showlegend=True,
        margin={'l': 1, 'r': 5, 't': 25, 'b': 1},
        # template='plotly_white',
    )


    #-------------------- Promociones + Descuentos ----------------------#

    PromDesc = Data[Data['PromocionDescuento']==1]

    fig.add_trace(
        go.Scatter(x=PromDesc['FechaVenta'],
                   y=yO.loc[PromDesc['FechaVenta']].values.flatten(),
                   name='Promocion-Descuentos',
                   marker={'color': 'firebrick',
                           'size': 10},
                   mode='markers',
                   )
    )

    #========================= PREDICCION ==========================#
    M = pd.DataFrame(yO.values,
                     index=yO.index,
                     columns=['Y'])
    A = yO.to_numpy().flatten().tolist()

    M['dia'] = M.index.day  # cifra para el dia de la Semana = 1, ... ,31
    M['mes'] = M.index.month
    M['año'] = M.index.year
    M['diaSemana'] = M.index.weekday  # nombre del dia de la Semana = Lunes, ... ,Domingo

    M['CantidadCerosUltimos7Dias'] = [A[max(0, i - 7): max(0, i)].count(0) for i in range(len(A))]
    M['NumeroDiasDesdeUltimaVenta'] = np.array([i - np.where(np.array(A[:i]) != 0)[0][-1] if i > np.where(np.array(A) != 0)[0][0]
                                                else i + 1 for i in range(len(A))])
    M['CantidadTotalVentaUltimos7Dias'] = [np.array(A[max(0, i - 7): max(0, i)]).sum() for i in range(len(A))]


    xgb_Regressor = xgb.XGBRegressor(max_depth=slid1,
                                     n_estimators=slid2)
    xgb_Regressor.fit(M[M.columns[1:]], M['Y'])

    Pred = mi_Prediccion(A,xgb_Regressor,730)
    Pred = pd.Series(Pred,index=pd.date_range(start='2024-01-01', freq='D', periods=730))

    fig.add_trace(
        go.Scatter(x=Pred.index,
                   y=Pred.values.flatten(),
                   name='Prediccion',
                   line={'color': 'darkorange',
                         'width': 1,
                         'dash': 'dashdot'},
                   marker={'color': 'darkorange',
                           'size': 6},
                   mode='lines+markers',
                   )
    )

    #------------------- Importancia Variables --------------------#

    #Importancia = pd.DataFrame( xgb_Regressor.feature_importances_.reshape(1,-1), columns=M.columns[1:] )  # feature_importance
    Fo = {i:xgb_Regressor.feature_importances_[n] for n,i in enumerate(M.columns[1:])}
    Fo_sort = dict(sorted(Fo.items(), key=lambda item: item[1], reverse=False))  # key=operator.itemgetter(1)


    fig_2 = go.Figure()
    fig_2.add_trace(
        go.Bar(y=list(Fo_sort.keys()),
               x=list(Fo_sort.values()),
               marker_color='steelblue',
               orientation='h')
    )

    fig_2.update_layout(
        title= 'Importancia de las Variables',
        title_x=0.25,  # center Title
        xaxis_title='',
        yaxis_title='',
        #xaxis_tickangle=30,
        showlegend=False,
        margin={'l': 10, 'r': 10, 't': 50, 'b': 10},
        # template='plotly_white',
    )
    fig_2.update_yaxes(tickfont_size=14)


    return(fig, fig_2)


# #========================================#
# #          Cuadro Prom-Descuento         #
# #========================================#
# @app.callback(Output(component_id='CuadroDescuento', component_property='figure'),
#               Input(component_id='Tienda', component_property='value'),
#               Input(component_id='Producto', component_property='value'))
# def Efecto_PromDescuento(t,p):
#     Data = Datos[(Datos['TiendaId'] == t) & (Datos['ProductoId'] == p)].sort_values(by='FechaVenta')
#
#     PromVenta = Data[['CantidadVendida', 'PrecioUnitario', 'PromocionDescuento']]
#     #print(PromVenta.head(10))
#
#     fig_1 = go.Figure()
#     fig_1.add_trace(
#         go.Scatter(x=PromVenta[PromVenta['PromocionDescuento']==0]['CantidadVendida'],
#                    y=PromVenta[PromVenta['PromocionDescuento']==0]['PrecioUnitario'],
#                    name='sin Promo-Desc',
#                    marker={'color': 'firebrick',
#                            'size': 10},
#                    mode='markers',
#                    )
#     )
#
#     fig_1.add_trace(
#         go.Scatter(x=PromVenta[PromVenta['PromocionDescuento'] == 1]['CantidadVendida'],
#                    y=PromVenta[PromVenta['PromocionDescuento'] == 1]['PrecioUnitario'],
#                    name='con Promo-Desc',
#                    marker={'color': 'darkgreen',
#                            'size': 10},
#                    mode='markers',
#                    )
#     )
#
#     fig_1.add_trace(
#         go.Scatter(x=[(PromVenta['CantidadVendida'].min()+PromVenta['CantidadVendida'].max())/2,
#                        1.05*PromVenta['CantidadVendida'].max()],
#                    y=[(PromVenta['PrecioUnitario'].min()+PromVenta['PrecioUnitario'].max())/2,
#                       (PromVenta['PrecioUnitario'].min()+PromVenta['PrecioUnitario'].max())/2],
#                    #mode='lines',
#                    name='',
#                    fill='tozeroy',
#                    fillcolor='rgba(0,100,0,0.2)',
#                    line={'color': 'rgba(0,0,0,0)',
#                          #'dash': 'dash'
#                          }
#                    )
#     )
#
#     #--------------------- fill to-next-Y ------------------#
#     fig_1.add_trace(
#         go.Scatter(x=[0.05 * PromVenta['CantidadVendida'].min(),
#                       (PromVenta['CantidadVendida'].min() + PromVenta['CantidadVendida'].max()) / 2],
#                    y=[1.05*PromVenta['PrecioUnitario'].max(),
#                       1.05*PromVenta['PrecioUnitario'].max()],
#                    # mode='lines',
#                    name='',
#                    line={'color': 'rgba(0,0,0,0)'}
#                    )
#     )
#     fig_1.add_trace(  ### plotted side-by-side
#         go.Scatter(x=[0.05*PromVenta['CantidadVendida'].min(),
#                       (PromVenta['CantidadVendida'].min()+PromVenta['CantidadVendida'].max())/2],
#                    y=[(PromVenta['PrecioUnitario'].min() + PromVenta['PrecioUnitario'].max()) / 2,
#                       (PromVenta['PrecioUnitario'].min() + PromVenta['PrecioUnitario'].max()) / 2],
#                    # mode='lines',
#                    name='',
#                    fill='tonexty',
#                    fillcolor='rgba(102,0,0,0.2)',
#                    line={'color': 'rgba(0,0,0,0)'}
#                    )
#     )
#     #---------------------------------------------------------#
#
#     fig_1.update_layout(
#         title='Efecto de Promociones-Descuento',
#         title_x=0.1,  # center Title
#         xaxis_title='Cantidad Vendida',
#         yaxis_title='Precio Unitario',
#         showlegend=True,
#         margin={'l': 1, 'r': 5, 't': 25, 'b': 1},
#         # template='plotly_white',
#     )
#
#     return(fig_1)




app.run_server()





C:\Users\guy3h\AppData\Local\Temp\ipykernel_24656\1910564870.py:8: UserWarning: 
The dash_html_components package is deprecated. Please replace
`import dash_html_components as html` with `from dash import html`
  import dash_html_components as html
C:\Users\guy3h\AppData\Local\Temp\ipykernel_24656\1910564870.py:9: UserWarning: 
The dash_core_components package is deprecated. Please replace
`import dash_core_components as dcc` with `from dash import dcc`
  import dash_core_components as dcc
C:\Users\guy3h\AppData\Local\Temp\ipykernel_24656\1910564870.py:14: UserWarning: 
The dash_table package is deprecated. Please replace
`import dash_table` with `from dash import dash_table`

Also, if you're using any of the table format helpers (e.g. Group), replace 
`from dash_table.Format import Group` with 
`from dash.dash_table.Format import Group`
  from dash_table.Format import Format


Dash is running on http://127.0.0.1:8050/

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:8050
Press CTRL+C to quit
127.0.0.1 - - [19/Mar/2026 08:29:30] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [19/Mar/2026 08:29:30] "GET /_dash-dependencies HTTP/1.1" 200 -
127.0.0.1 - - [19/Mar/2026 08:29:30] "GET /_dash-layout HTTP/1.1" 200 -
127.0.0.1 - - [19/Mar/2026 08:29:30] "GET /_dash-component-suites/dash/dash_table/async-highlight.js HTTP/1.1" 304 -
127.0.0.1 - - [19/Mar/2026 08:29:30] "GET /_dash-component-suites/dash/dcc/async-graph.js HTTP/1.1" 304 -
127.0.0.1 - - [19/Mar/2026 08:29:30] "GET /_dash-component-suites/dash/dcc/async-dropdown.js HTTP/1.1" 304 -
127.0.0.1 - - [19/Mar/2026 08:29:30] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [19/Mar/2026 08:29:31] "POST /_dash-update-component HTTP/1.1" 200 -
[2026-03-19 08:29:31,105] ERROR in app: Exception on /_dash-update-component [POST]
Traceback (most recent call last):
  File "C:\Users\guy3h\AppData\Local\Programs\Python\Python311\Lib\site-packages\flask\app.py", line 2528, in wsgi_a